In [19]:
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.
    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются
    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [20]:
from pathlib import Path

DOCS_DIR = Path("docs")

# Загрузка и чанкинг - это вы уже видели в теории
chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

Всего чанков: 543


In [3]:
# 1. Сколько разных файлов дали чанки?
unique_sources = {c["source"] for c in chunks}
print(f"Файлов с чанками: {len(unique_sources)}")

Файлов с чанками: 61


In [4]:
# Counter из модуля collections умеет считать частоты
from collections import Counter

# 2. Сколько чанков пришло из каждого файла?
#    Подсказка: соберите Counter по c["source"] для всех чанков
source_counts = Counter(c["source"] for c in chunks)
for i in source_counts:
    print(f'{source_counts[i]}\t{i}')

2	docs/README.md
19	docs/api/anthropic-compatibility.mdx
4	docs/api/authentication.mdx
3	docs/api/errors.mdx
6	docs/api/introduction.mdx
20	docs/api/openai-compatibility.mdx
4	docs/api/streaming.mdx
2	docs/api/usage.mdx
72	docs/api.md
7	docs/capabilities/embeddings.mdx
7	docs/capabilities/streaming.mdx
11	docs/capabilities/structured-outputs.mdx
8	docs/capabilities/thinking.mdx
33	docs/capabilities/tool-calling.mdx
5	docs/capabilities/vision.mdx
19	docs/capabilities/web-search.mdx
3	docs/cli.mdx
7	docs/cloud.mdx
3	docs/context-length.mdx
13	docs/development.md
8	docs/docker.mdx
4	docs/examples.md
37	docs/faq.mdx
14	docs/gpu.mdx
5	docs/import.mdx
4	docs/index.mdx
7	docs/integrations/chatgpt.mdx
14	docs/integrations/claude-code.mdx
6	docs/integrations/claude-desktop.mdx
4	docs/integrations/cline-cli.mdx
3	docs/integrations/cline.mdx
4	docs/integrations/codex.mdx
6	docs/integrations/copilot-cli.mdx
5	docs/integrations/deepseek-harness.mdx
4	docs/integrations/droid.mdx
2	docs/integrations/

In [5]:
# 3. Топ-3 файла по числу чанков
top_3 = source_counts.most_common(3)
print("Топ-3 файла по числу чанков:")
for source, count in top_3:
    print(f"  {source}: {count} чанков")

Топ-3 файла по числу чанков:
  docs/api.md: 72 чанков
  docs/faq.mdx: 37 чанков
  docs/capabilities/tool-calling.mdx: 33 чанков


In [6]:
sample_text = Path("docs/api.md").read_text(encoding="utf-8")

for max_chars in [200, 800, 3000]:
    chunks_sample = chunk_text(sample_text, max_chars=max_chars)
    # Ваш код здесь: посчитайте количество чанков и среднюю длину
    # Выведите результат в формате: max_chars=200 -> чанков: X, средняя длина: Y
    print(f'>>> max_chars = {max_chars}\t{len(chunks_sample)}')

    average_length = sum(len(s) for s in chunks_sample) / len(chunks_sample) 
    print(int(average_length))

>>> max_chars = 200	159
328
>>> max_chars = 800	72
742
>>> max_chars = 3000	30
1785


In [7]:
sample_text = Path("docs/api.md").read_text(encoding="utf-8")

chunks_sample = chunk_text(sample_text, max_chars=200)

print(chunks_sample[5])

## Generate a completion

```
POST /api/generate
```


In [12]:
sample_text = Path("docs/api.md").read_text(encoding="utf-8")

chunks_sample = chunk_text(sample_text, max_chars=800)

print(chunks_sample[5])

#### Structured outputs

Structured outputs are supported by providing a JSON schema in the `format` parameter. The model will generate a response that matches the schema. See the [structured outputs](#request-structured-outputs) example below.

#### JSON mode

Enable JSON mode by setting the `format` parameter to `json`. This will structure the response as a valid JSON object. See the JSON mode [example](#request-json-mode) below.

> [!IMPORTANT]
> It's important to instruct the model to use JSON in the `prompt`. Otherwise, the model may generate large amounts whitespace.

### Examples

#### Generate request (Streaming)

##### Request

```shell
curl http://localhost:11434/api/generate -d '{
  "model": "llama3.2",
  "prompt": "Why is the sky blue?"
}'
```

##### Response


In [9]:
sample_text = Path("docs/api.md").read_text(encoding="utf-8")

chunks_sample = chunk_text(sample_text, max_chars=3000)

print(chunks_sample[5])

```shell
curl -X POST http://localhost:11434/api/generate -H "Content-Type: application/json" -d '{
  "model": "llama3.1:8b",
  "prompt": "Ollama is 22 years old and is busy saving the world. Respond using JSON",
  "stream": false,
  "format": {
    "type": "object",
    "properties": {
      "age": {
        "type": "integer"
      },
      "available": {
        "type": "boolean"
      }
    },
    "required": [
      "age",
      "available"
    ]
  }
}'
```

##### Response

```json
{
  "model": "llama3.1:8b",
  "created_at": "2024-12-06T00:48:09.983619Z",
  "response": "{\n  \"age\": 22,\n  \"available\": true\n}",
  "done": true,
  "done_reason": "stop",
  "context": [1, 2, 3],
  "total_duration": 1075509083,
  "load_duration": 567678166,
  "prompt_eval_count": 28,
  "prompt_eval_duration": 236000000,
  "eval_count": 16,
  "eval_duration": 269000000
}
```

#### Request (JSON mode)

> [!IMPORTANT]
> When `format` is set to `json`, the output will always be a well-formed JSON object

In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)

# Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

Загружаем модель эмбеддингов: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Размер матрицы эмбеддингов: (543, 384)


In [14]:
print(f"Первый чанк: {chunks[0]['text'][:80]}...")
print(f"Его эмбеддинг (первые 10 чисел): {chunk_embeddings[0][:10]}")
print(f"Длина вектора (норма): {np.linalg.norm(chunk_embeddings[0]):.4f}")

Первый чанк: # Documentation

### Getting Started
* [Quickstart](https://docs.ollama.com/quic...
Его эмбеддинг (первые 10 чисел): [-0.06130742 -0.05510944 -0.03115823 -0.0100545   0.03433185  0.04836936
 -0.08265638  0.02022991 -0.00097116  0.04949648]
Длина вектора (норма): 1.0000


In [15]:
import faiss

dim = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print(f"В индексе {index.ntotal} векторов размерности {dim}")

В индексе 543 векторов размерности 384


In [16]:
def vector_search(question: str, top_k: int = 3) -> list[dict]:
    """Возвращает top_k чанков, наиболее близких к запросу по смыслу."""
    # 1. Считаем эмбеддинг запроса
    query_emb = embed_model.encode(
        [question],
        normalize_embeddings=True,
    ).astype(np.float32)

    # 2. Ищем ближайших соседей
    scores, indices = index.search(query_emb, top_k)

    # 3. Собираем результат с метаданными
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(score),
        })
    return results

In [17]:
question = "Как сменить папку, в которой Ollama хранит модели?"
results = vector_search(question, top_k=3)

print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

Запрос: Как сменить папку, в которой Ollama хранит модели?

#1  [docs/windows.mdx]  score=0.765
To change where Ollama stores the downloaded models instead of using your home directory, set the environment variable `OLLAMA_MODELS` in your user account.

1. Start the Settings (Windows 11) or Cont ...

#2  [docs/macos.mdx]  score=0.629
## Filesystem Requirements

The preferred method of installation is to mount the `ollama.dmg` and drag-and-drop the Ollama application to the system-wide `Applications` folder.  Upon startup, the Olla ...

#3  [docs/faq.mdx]  score=0.620
Alternatively, you can change the amount of time all models are loaded into memory by setting the `OLLAMA_KEEP_ALIVE` environment variable when starting the Ollama server. The `OLLAMA_KEEP_ALIVE` vari ...



In [ ]:
# Задание 2 из 2

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)

# Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

# 3. build index
dim = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print(f"В индексе {index.ntotal} векторов размерности {dim}")

Загружаем модель эмбеддингов: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Размер матрицы эмбеддингов: (543, 384)
В индексе 543 векторов размерности 384


In [27]:
# 4.
question = "What HTTP method does /api/generate use?"
results = vector_search(question, top_k=3)

print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

Запрос: What HTTP method does /api/generate use?

#1  [docs/faq.mdx]  score=0.461
```shell
docker build -t ollama-with-ca .
docker run -d -e HTTPS_PROXY=https://my.proxy.example.com -p 11434:11434 ollama-with-ca
``` ...

#2  [docs/quickstart.mdx]  score=0.454
These examples use the `gemma4:31b` [cloud model](https://ollama.com/search?c=cloud).

<Tabs>
  <Tab title="Ollama API">

```shell
curl https://ollama.com/api/chat \
  -H "Authorization: Bearer $OLLAM ...

#3  [docs/api/introduction.mdx]  score=0.453
## Base URLs

| API | Direct cloud access | Local Ollama server |
| --- | --- | --- |
| Ollama | `https://ollama.com/api` | `http://localhost:11434/api` |
| OpenAI compatibility | `https://ollama.com/ ...



In [28]:
question = "Как сменить папку, где хранятся модели?"
results = vector_search(question, top_k=3)

print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

Запрос: Как сменить папку, где хранятся модели?

#1  [docs/faq.mdx]  score=0.586
## Where are models stored?

- macOS: `~/.ollama/models`
- Linux: `/usr/share/ollama/.ollama/models`
- Windows: `C:\Users\%username%\.ollama\models`

### How do I set them to a different location?

If ...

#2  [docs/windows.mdx]  score=0.581
To change where Ollama stores the downloaded models instead of using your home directory, set the environment variable `OLLAMA_MODELS` in your user account.

1. Start the Settings (Windows 11) or Cont ...

#3  [docs/integrations/claude-desktop.mdx]  score=0.502
## Switch models

Open Ollama **Settings → Apps**. Choose your models, then select **Restart Claude**. You can use the same model for more than one option. ...



In [34]:
question = "What is a Modelfile?"
results = vector_search(question, top_k=3)

print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

RecursionError: maximum recursion depth exceeded